# EDA y Modelado con Scikit-learn + Pandas

Este notebook replica el pipeline académico original (Pima Indians Diabetes) en el ecosistema **pandas / scikit-learn**, y añade tres bloques nuevos:

1. **Análisis ROC** — curvas ROC y AUC para todos los modelos.
2. **Detección de outliers con LOF** (*Local Outlier Factor*) — método multivariante basado en densidad local.
3. **Tratamiento de datasets desbalanceados** — submuestreo aleatorio, sobremuestreo aleatorio y **SMOTE** (con `imbalanced-learn`).

**Dataset:** `pima-indians-diabetes.csv`
**Target:** `diabetes` (0 = sano, 1 = diabético)


## 1. Configuración del entorno

In [ ]:
# Instalación (Colab): imbalanced-learn no viene preinstalado en algunos entornos
!pip install -q imbalanced-learn scikit-learn pandas matplotlib seaborn

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# scikit-learn
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, GridSearchCV, cross_val_score
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report,
    ConfusionMatrixDisplay
)

# imbalanced-learn
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline   # acepta samplers en pipeline

# Semilla global para reproducibilidad
SEED = 1234
np.random.seed(SEED)

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 100

## 2. Carga del dataset

### Camino 1 — Carga simple con inferencia automática de tipos


In [ ]:
# Equivalente a spark.read.csv(..., inferSchema=True)
df_base = pd.read_csv("pima-indians-diabetes.csv")

df_base.head(5)

In [ ]:
# Tipos de datos (equivalente a printSchema)
print(df_base.dtypes)

In [ ]:
# Dimensiones del dataframe
print(f"Filas: {df_base.shape[0]}  |  Columnas: {df_base.shape[1]}")

In [ ]:
# Estadísticas descriptivas
df_base.describe().round(3)

In [ ]:
# Distribución de la variable objetivo
df_base["diabetes"].value_counts()

### Camino 2 — Carga con esquema explícito

En pandas no hay un concepto formal de *schema* como en Spark, pero podemos forzar el tipo de cada columna con el parámetro `dtype` de `read_csv`. Es buena práctica en producción para detectar tempranamente entradas mal formadas.


In [ ]:
schema = {
    "embarazos":            "float64",
    "concentracionGlucosa": "float64",
    "presionArterial":      "float64",
    "pliegueCutaneo":       "float64",
    "insulinaSerica":       "float64",
    "IMC":                  "float64",
    "funcionPedigree":      "float64",
    "edad":                 "float64",
    "diabetes":             "int64",
}

FILE_PATH = "pima-indians-diabetes.csv"

df_raw = pd.read_csv(FILE_PATH, dtype=schema)

print(f"Filas: {df_raw.shape[0]}  |  Columnas: {df_raw.shape[1]}")
print("\nTipos:")
print(df_raw.dtypes)

In [ ]:
# Primeras filas
df_raw.head(10)

In [ ]:
# Estadísticas descriptivas con .describe()
desc_pd = df_raw.describe().T
print(desc_pd.round(3).to_string())

## 3. Distribución de la variable objetivo

In [ ]:
# Conteo y porcentajes
total = len(df_raw)
class_pd = (df_raw["diabetes"]
            .value_counts()
            .rename_axis("diabetes")
            .reset_index(name="conteo"))
class_pd["porcentaje"] = (class_pd["conteo"] / total * 100).round(2)
class_pd = class_pd.sort_values("diabetes").reset_index(drop=True)
print(class_pd)

In [ ]:
# Gráfico de distribución
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(["Sano (0)", "Diabético (1)"], class_pd["conteo"],
       color=["steelblue", "tomato"], edgecolor="black")
ax.set_title("Distribución de la variable objetivo (diabetes)", fontsize=13)
ax.set_ylabel("Cantidad de registros")
for i, v in enumerate(class_pd["conteo"]):
    ax.text(i, v + 5, f"{v}\n({class_pd['porcentaje'].iloc[i]}%)",
            ha="center", fontsize=11)
plt.tight_layout()
plt.show()

## 4. EDA — Histogramas, asimetría y curtosis

In [ ]:
feature_cols = [c for c in df_raw.columns if c != "diabetes"]

fig, axes = plt.subplots(nrows=3, ncols=3, figsize=(15, 10))
axes = axes.flatten()
for i, col in enumerate(df_raw.columns):
    color = "tomato" if col == "diabetes" else "steelblue"
    axes[i].hist(df_raw[col].dropna(), bins=20, color=color,
                 edgecolor="black", alpha=0.8)
    axes[i].set_title(col, fontsize=11)
    axes[i].set_xlabel("Valor"); axes[i].set_ylabel("Frecuencia")
plt.suptitle("Histogramas de todas las variables", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Skewness (asimetría) — pandas usa la fórmula ajustada de Fisher-Pearson
skew_pd = df_raw.skew(numeric_only=True).round(4)
print("Valores de Skewness (Asimetría):")
print(skew_pd.to_string())

In [ ]:
# Curtosis (exceso de Fisher) — equivalente a F.kurtosis() de PySpark
kurt_pd = pd.DataFrame({
    "variable": feature_cols,
    "curtosis": [round(df_raw[c].kurt(), 4) for c in feature_cols]
}).sort_values("curtosis", ascending=False)

def clasificar_curtosis(k):
    if k > 1:   return "Leptocúrtica"
    if k < -1:  return "Platicúrtica"
    return "Mesocúrtica"

kurt_pd["tipo"] = kurt_pd["curtosis"].apply(clasificar_curtosis)

print("=" * 50)
print("Curtosis por variable (exceso de Fisher)")
print("=" * 50)
print(kurt_pd.to_string(index=False))

In [ ]:
# Visualización de la curtosis
color_map = {"Leptocúrtica": "tomato", "Mesocúrtica": "steelblue", "Platicúrtica": "orange"}

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(kurt_pd["variable"], kurt_pd["curtosis"],
              color=[color_map[t] for t in kurt_pd["tipo"]],
              edgecolor="black", alpha=0.85)

ax.axhline(0,  color="black",  linewidth=1.2, linestyle="-",  label="Normal (0)")
ax.axhline(1,  color="tomato", linewidth=1,   linestyle="--", label="Umbral Leptocúrtica (+1)")
ax.axhline(-1, color="orange", linewidth=1,   linestyle="--", label="Umbral Platicúrtica (-1)")

for bar, val in zip(bars, kurt_pd["curtosis"]):
    va = "bottom" if val >= 0 else "top"
    offset = 0.05 if val >= 0 else -0.05
    ax.text(bar.get_x() + bar.get_width()/2, val + offset,
            f"{val:.2f}", ha="center", va=va, fontsize=9, fontweight="bold")

ax.set_xlabel("Variable"); ax.set_ylabel("Curtosis (exceso)")
ax.set_title("Curtosis por variable", fontsize=13)
ax.legend(fontsize=9)
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## 5. Matriz de correlación (Pearson)

In [ ]:
corr_pd = df_raw.corr(method="pearson")

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_pd, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, vmin=-1, vmax=1, ax=ax, annot_kws={"size": 8})
ax.set_title("Mapa de calor — Correlación de Pearson", fontsize=13)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 6. Boxplots por clase — separabilidad de variables

In [ ]:
# Muestreo (en pandas opcional; el dataset es pequeño)
pdf_muestra = df_raw[["diabetes", "edad", "IMC", "concentracionGlucosa"]].sample(
    frac=0.25, random_state=123
)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
sns.boxplot(ax=axes[0], x="diabetes", y="edad",                data=pdf_muestra, palette="Set2")
axes[0].set_title("Distribución de Edad")
sns.boxplot(ax=axes[1], x="diabetes", y="IMC",                 data=pdf_muestra, palette="Set2")
axes[1].set_title("Distribución de IMC")
sns.boxplot(ax=axes[2], x="diabetes", y="concentracionGlucosa", data=pdf_muestra, palette="Set2")
axes[2].set_title("Distribución de Glucosa")
plt.tight_layout()
plt.show()

In [ ]:
# Boxplots por clase para todas las features
fig, axes = plt.subplots(nrows=2, ncols=4, figsize=(16, 8))
axes = axes.flatten()
for i, col in enumerate(feature_cols):
    df_raw.boxplot(column=col, by="diabetes", ax=axes[i],
                   boxprops=dict(color="steelblue"),
                   medianprops=dict(color="tomato", linewidth=2))
    axes[i].set_title(col, fontsize=10)
    axes[i].set_xlabel("Diabetes (0=Sano, 1=Diabético)")
plt.suptitle("Boxplots por clase — separabilidad de variables", fontsize=13)
plt.tight_layout()
plt.show()

## 7. Tratamiento de valores faltantes

En el dataset Pima, varias columnas usan `0` para representar valores ausentes (médicamente imposibles). Primero los reemplazamos por `NaN` y luego comparamos cinco estrategias clásicas.


In [ ]:
COLS_CON_CEROS_INVALIDOS = [
    "concentracionGlucosa",
    "presionArterialSistolica",
    "pliegueCutaneo",
    "insulinaSerica",
    "IMC",
]

# Reemplazar 0 → NaN en las columnas afectadas
df_nulls = df_raw.copy()
df_nulls[COLS_CON_CEROS_INVALIDOS] = df_nulls[COLS_CON_CEROS_INVALIDOS].replace(0, np.nan)

# Conteo de nulos por columna
null_counts = df_nulls.isna().sum()
print("Nulos por columna:")
print(null_counts)

In [ ]:
# % de nulos por columna
null_pd = (null_counts[null_counts > 0]
           .rename("nulos")
           .reset_index()
           .rename(columns={"index": "columna"}))
null_pd["pct"] = (null_pd["nulos"] / len(df_raw) * 100).round(2)
null_pd = null_pd.sort_values("pct", ascending=False)

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(null_pd["columna"], null_pd["pct"], color="coral", edgecolor="black")
ax.set_xlabel("% de valores faltantes")
ax.set_title("Porcentaje de valores faltantes por columna")
for bar, pct in zip(bars, null_pd["pct"]):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f"{pct}%", va="center")
plt.tight_layout()
plt.show()

### Estrategia A — Eliminar filas con nulos

In [ ]:
df_dropna = df_nulls.dropna()
print(f"Filas originales : {len(df_raw)}")
print(f"Filas tras dropna: {len(df_dropna)}")
print(f"Filas eliminadas : {len(df_raw) - len(df_dropna)}")

### Estrategia B — Eliminar columnas con más del 25% de nulos

In [ ]:
threshold = 0.25
null_fracs = df_nulls.isna().mean()
cols_to_drop = null_fracs[null_fracs > threshold].index.tolist()

print(f"Columnas eliminadas (>{threshold*100:.0f}% nulos): {cols_to_drop}")

df_dropcols = df_nulls.drop(columns=cols_to_drop)
print(f"Columnas restantes: {list(df_dropcols.columns)}")

### Estrategia C — Imputar con la media (`SimpleImputer`)

In [ ]:
imputer_mean = SimpleImputer(strategy="mean")
df_imputed_mean = df_nulls.copy()
df_imputed_mean[COLS_CON_CEROS_INVALIDOS] = imputer_mean.fit_transform(
    df_nulls[COLS_CON_CEROS_INVALIDOS]
)

# Verificar que no quedan nulos
print("Nulos restantes por columna:")
print(df_imputed_mean[COLS_CON_CEROS_INVALIDOS].isna().sum())

print("\nMedias usadas para imputación:")
for col, mean_val in zip(COLS_CON_CEROS_INVALIDOS, imputer_mean.statistics_):
    print(f"  {col}: {mean_val:.4f}")

### Estrategia D — Imputar con la mediana

In [ ]:
imputer_median = SimpleImputer(strategy="median")
df_imputed_median = df_nulls.copy()
df_imputed_median[COLS_CON_CEROS_INVALIDOS] = imputer_median.fit_transform(
    df_nulls[COLS_CON_CEROS_INVALIDOS]
)

print("Medianas usadas para imputación:")
for col, med_val in zip(COLS_CON_CEROS_INVALIDOS, imputer_median.statistics_):
    print(f"  {col}: {med_val:.4f}")

### Estrategia E — Forward-fill (`ffill`)

Análogo a la imputación con *Window functions* en PySpark.


In [ ]:
df_ffill = df_nulls.copy()
df_ffill[COLS_CON_CEROS_INVALIDOS] = df_ffill[COLS_CON_CEROS_INVALIDOS].ffill()
df_ffill = df_ffill.dropna(subset=COLS_CON_CEROS_INVALIDOS)   # primeras filas sin valor previo
print(f"Filas tras forward-fill: {len(df_ffill)}")

## 8. Detección y manejo de outliers

A partir de aquí trabajamos sobre `df_imputed_mean` (imputado con media), siguiendo la misma elección del notebook original.


### Método 1 — Regla de 3 desviaciones estándar (3-sigma)

In [ ]:
std_bounds = {}
for col in feature_cols:
    mu, sigma = df_imputed_mean[col].mean(), df_imputed_mean[col].std()
    std_bounds[col] = {"lower": mu - 3*sigma, "upper": mu + 3*sigma}

# Construir máscara compuesta
mask_std = pd.Series(True, index=df_imputed_mean.index)
for col, b in std_bounds.items():
    mask_std &= df_imputed_mean[col].between(b["lower"], b["upper"])

df_no_outliers_std = df_imputed_mean[mask_std]
total = len(df_imputed_mean)
print(f"Filas originales              : {total}")
print(f"Filas sin outliers (3-sigma)  : {len(df_no_outliers_std)}")

### Método 2 — Rango Intercuartílico (IQR)

In [ ]:
iqr_bounds = {}
for col in feature_cols:
    q1, q3 = df_imputed_mean[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    iqr_bounds[col] = {
        "Q1": q1, "Q3": q3,
        "lower": q1 - 1.5*iqr,
        "upper": q3 + 1.5*iqr,
    }

print(f"{'Columna':<25} {'Q1':>8} {'Q3':>8} {'Lower':>10} {'Upper':>10}")
print("-" * 65)
for col, b in iqr_bounds.items():
    print(f"{col:<25} {b['Q1']:>8.2f} {b['Q3']:>8.2f} {b['lower']:>10.2f} {b['upper']:>10.2f}")

In [ ]:
# Filtrar outliers IQR
mask_iqr = pd.Series(True, index=df_imputed_mean.index)
for col, b in iqr_bounds.items():
    mask_iqr &= df_imputed_mean[col].between(b["lower"], b["upper"])

df_no_outliers_iqr = df_imputed_mean[mask_iqr]
print(f"Filas sin outliers (IQR)     : {len(df_no_outliers_iqr)}")
print(f"Filas sin outliers (3-sigma) : {len(df_no_outliers_std)}")

In [ ]:
# Boxplots antes/después
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, (df_plot, titulo) in zip(axes, [
    (df_imputed_mean,     "Con outliers (imputación media)"),
    (df_no_outliers_iqr,  "Sin outliers (IQR)")
]):
    df_plot[feature_cols].boxplot(ax=ax, rot=45)
    ax.set_title(titulo, fontsize=12)
plt.tight_layout()
plt.show()

### Método 3 — Local Outlier Factor (LOF) **[NUEVO]**

`LOF` es un detector **multivariante** basado en densidad: compara la densidad local de cada punto con la de sus vecinos. Un punto es atípico cuando su densidad local es significativamente menor que la de sus vecinos.

Ventajas frente a 3-sigma e IQR:
- No asume normalidad ni se aplica variable a variable: detecta anomalías **conjuntas** (combinaciones raras de varias variables).
- Funciona bien en datasets con clusters de densidad heterogénea.

Parámetros clave:
- `n_neighbors`: número de vecinos para estimar densidad local (típico 20).
- `contamination`: proporción esperada de outliers (puede ser `"auto"`).


In [ ]:
# LOF debe trabajar sobre datos estandarizados (las distancias son sensibles a la escala)
from sklearn.preprocessing import StandardScaler as _Scaler

X_for_lof = df_imputed_mean[feature_cols].copy()
X_scaled  = _Scaler().fit_transform(X_for_lof)

lof = LocalOutlierFactor(n_neighbors=20, contamination=0.1, novelty=False)
lof_labels = lof.fit_predict(X_scaled)          #  1 = inlier, -1 = outlier
lof_scores = -lof.negative_outlier_factor_      #  cuanto mayor, más anómalo

# Resumen
n_out = (lof_labels == -1).sum()
print(f"Filas etiquetadas como outlier por LOF: {n_out} de {len(lof_labels)} "
      f"({n_out/len(lof_labels)*100:.2f}%)")

In [ ]:
# Distribución de scores LOF
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(lof_scores, bins=40, color="steelblue", edgecolor="black", alpha=0.85)
ax.axvline(np.quantile(lof_scores, 0.9), color="tomato", linestyle="--",
           label="Percentil 90")
ax.set_xlabel("LOF score (más alto = más anómalo)")
ax.set_ylabel("Frecuencia")
ax.set_title("Distribución de scores LOF")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Dataset sin outliers LOF
df_no_outliers_lof = df_imputed_mean[lof_labels == 1].copy()
print(f"Filas conservadas tras LOF: {len(df_no_outliers_lof)}")

# Visualización 2D usando las dos variables más informativas: glucosa vs IMC
fig, ax = plt.subplots(figsize=(8, 6))
inliers  = lof_labels ==  1
outliers = lof_labels == -1
ax.scatter(df_imputed_mean.loc[inliers,  "concentracionGlucosa"],
           df_imputed_mean.loc[inliers,  "IMC"],
           s=20, c="steelblue", alpha=0.6, label="Inlier")
ax.scatter(df_imputed_mean.loc[outliers, "concentracionGlucosa"],
           df_imputed_mean.loc[outliers, "IMC"],
           s=60, c="tomato", edgecolor="black", label="Outlier LOF")
ax.set_xlabel("Concentración de Glucosa"); ax.set_ylabel("IMC")
ax.set_title("Outliers detectados por LOF\n(proyección 2D Glucosa vs IMC)")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Comparación numérica de los tres métodos
print("=" * 55)
print("COMPARACIÓN DE MÉTODOS DE DETECCIÓN DE OUTLIERS")
print("=" * 55)
print(f"{'Método':<15} {'Filas conservadas':>20} {'% conservado':>15}")
print("-" * 55)
n0 = len(df_imputed_mean)
for nombre, df_x in [("3-sigma", df_no_outliers_std),
                     ("IQR",     df_no_outliers_iqr),
                     ("LOF",     df_no_outliers_lof)]:
    print(f"{nombre:<15} {len(df_x):>20} {len(df_x)/n0*100:>14.2f}%")

## 9. d de Cohen — Tamaño del efecto por variable

In [ ]:
def cohen_d_pandas(df, feature_cols, group_col="diabetes"):
    """
    d = (mean_1 - mean_0) / std_pooled
    std_pooled = sqrt(((n1-1)*var1 + (n0-1)*var0) / (n1+n0-2))
    """
    results = []
    for col in feature_cols:
        g0 = df.loc[df[group_col] == 0, col].dropna()
        g1 = df.loc[df[group_col] == 1, col].dropna()
        n0, n1 = len(g0), len(g1)
        var0, var1 = g0.var(ddof=1), g1.var(ddof=1)
        std_pooled = np.sqrt(((n1-1)*var1 + (n0-1)*var0) / (n1+n0-2))
        d = (g1.mean() - g0.mean()) / std_pooled if std_pooled else 0.0
        abs_d = abs(d)
        if   abs_d < 0.2: efecto = "Insignificante"
        elif abs_d < 0.5: efecto = "Pequeño"
        elif abs_d < 0.8: efecto = "Mediano"
        else:             efecto = "Grande"
        results.append({
            "variable": col,
            "media_sano": round(g0.mean(), 4),
            "media_diabetico": round(g1.mean(), 4),
            "std_pooled": round(std_pooled, 4),
            "d_cohen": round(d, 4),
            "abs_d":   round(abs_d, 4),
            "efecto":  efecto,
        })
    return pd.DataFrame(results).sort_values("abs_d", ascending=False)

cohen_pd = cohen_d_pandas(df_imputed_mean, feature_cols)
print("=" * 75)
print("d de Cohen por variable (Diabético vs Sano)")
print("=" * 75)
print(cohen_pd.to_string(index=False))

In [ ]:
# Visualización
fig, ax = plt.subplots(figsize=(10, 6))
colors_map = {"Grande": "tomato", "Mediano": "orange",
              "Pequeño": "steelblue", "Insignificante": "lightgrey"}
colors = cohen_pd["efecto"].map(colors_map)

bars = ax.barh(cohen_pd["variable"][::-1], cohen_pd["abs_d"][::-1],
               color=colors[::-1], edgecolor="black")
for umbral, label, ls in [(0.2, "pequeño", "--"),
                          (0.5, "mediano", "-."),
                          (0.8, "grande", ":")]:
    ax.axvline(x=umbral, color="black", linestyle=ls, alpha=0.6,
               label=f"|d|={umbral} ({label})")

ax.set_xlabel("d de Cohen (valor absoluto)")
ax.set_title("Tamaño del efecto por variable — d de Cohen\n(Diabético vs Sano)",
             fontsize=13)
ax.legend(fontsize=9)
for bar, val, efecto in zip(bars, cohen_pd["abs_d"][::-1], cohen_pd["efecto"][::-1]):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
            f"{val:.3f} ({efecto})", va="center", fontsize=9)
plt.tight_layout()
plt.show()

## 10. Distribución KDE por clase

In [ ]:
df_kde = df_imputed_mean.copy()
df_kde["diabetes"] = df_kde["diabetes"].map({0: "Sano", 1: "Diabético"})
palette = {"Sano": "steelblue", "Diabético": "tomato"}

fig, axes = plt.subplots(nrows=2, ncols=4, figsize=(18, 9))
axes = axes.flatten()
for i, col in enumerate(feature_cols):
    ax = axes[i]
    for clase, color in palette.items():
        subset = df_kde[df_kde["diabetes"] == clase][col].dropna()
        sns.kdeplot(subset, ax=ax, label=clase, color=color,
                    fill=True, alpha=0.35, linewidth=2)
        ax.axvline(subset.mean(), color=color, linestyle="--",
                   linewidth=1.5, alpha=0.8)
    row_cohen = cohen_pd[cohen_pd["variable"] == col]
    if not row_cohen.empty:
        d_val   = row_cohen["d_cohen"].values[0]
        efecto  = row_cohen["efecto"].values[0]
        ax.set_title(f"{col}\nd de Cohen = {d_val:.3f} ({efecto})", fontsize=10)
    else:
        ax.set_title(col, fontsize=10)
    ax.set_xlabel("Valor"); ax.set_ylabel("Densidad"); ax.legend(fontsize=8)

for j in range(len(feature_cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Distribución KDE por variable — Diabético vs Sano\n"
             "(líneas punteadas = media de cada grupo)", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 11. Preparación para modelado

### Split estratificado

`sklearn` ofrece `train_test_split(..., stratify=y)` directamente.


In [ ]:
# Dataset A: imputado con media (incluye outliers)
df_A = df_imputed_mean.copy()
X_A = df_A[feature_cols]
y_A = df_A["diabetes"].astype(int)

X_train_A, X_test_A, y_train_A, y_test_A = train_test_split(
    X_A, y_A, test_size=0.2, stratify=y_A, random_state=SEED
)

# Dataset B: imputado con media + sin outliers IQR
df_B = df_no_outliers_iqr.copy()
X_B = df_B[feature_cols]
y_B = df_B["diabetes"].astype(int)

X_train_B, X_test_B, y_train_B, y_test_B = train_test_split(
    X_B, y_B, test_size=0.2, stratify=y_B, random_state=SEED
)

print("Dataset A (con outliers):")
print(f"  Train: {len(X_train_A)} filas | Test: {len(X_test_A)} filas")
print("\nDataset B (sin outliers IQR):")
print(f"  Train: {len(X_train_B)} filas | Test: {len(X_test_B)} filas")

### Pipeline de preprocesamiento

Combinamos `StandardScaler` (no necesario para árboles, pero sí buena práctica) con el clasificador. 

In [ ]:
def build_preprocessing_pipeline(classifier):
    """Pipeline scaler + clasificador."""
    return Pipeline(steps=[
        ("scaler", StandardScaler(with_mean=True, with_std=True)),
        ("clf",    classifier),
    ])

## 12. Modelado — Árbol de Decisión y Random Forest

In [ ]:
def evaluar_modelo(model, X_test, y_test, nombre):
    """Calcula Accuracy, F1, Precision, Recall, AUC-ROC y devuelve probabilidades."""
    y_pred = model.predict(X_test)
    # AUC requiere las probabilidades de la clase positiva
    y_proba = model.predict_proba(X_test)[:, 1]

    metrics = {
        "Modelo":     nombre,
        "Accuracy":   round(accuracy_score(y_test, y_pred), 4),
        "F1":         round(f1_score(y_test, y_pred, average="weighted"), 4),
        "Precision":  round(precision_score(y_test, y_pred, average="weighted"), 4),
        "Recall":     round(recall_score(y_test, y_pred, average="weighted"), 4),
        "AUC-ROC":    round(roc_auc_score(y_test, y_proba), 4),
    }
    conf = confusion_matrix(y_test, y_pred)
    return metrics, conf, y_pred, y_proba


def train_full_pipeline(X_train, y_train, X_test, y_test, classifier, nombre_modelo):
    """Entrena pipeline (scaler + clasificador) y evalúa sobre test."""
    pipe = build_preprocessing_pipeline(classifier)
    print(f"\n{'='*55}")
    print(f"Entrenando: {nombre_modelo}")
    print('='*55)
    pipe.fit(X_train, y_train)
    metrics, conf, y_pred, y_proba = evaluar_modelo(pipe, X_test, y_test, nombre_modelo)

    for k, v in metrics.items():
        if k != "Modelo":
            print(f"  {k:<12}: {v}")

    print("\nMatriz de confusión:")
    print(conf)

    return pipe, metrics, y_pred, y_proba

In [ ]:
results = {}        # métricas
roc_data = {}       # (y_test, y_proba) para curvas ROC

# ── Dataset A (con outliers) ────────────────────────────────────────────────
model_dt_A, metrics_dt_A, _, proba_dt_A = train_full_pipeline(
    X_train_A, y_train_A, X_test_A, y_test_A,
    DecisionTreeClassifier(max_depth=10, min_samples_leaf=5, random_state=SEED),
    "Árbol de Decisión — Dataset A (con outliers)"
)
results["DT_A"]  = metrics_dt_A
roc_data["DT_A"] = (y_test_A, proba_dt_A)

model_rf_A, metrics_rf_A, _, proba_rf_A = train_full_pipeline(
    X_train_A, y_train_A, X_test_A, y_test_A,
    RandomForestClassifier(n_estimators=100, max_depth=10,
                           min_samples_leaf=5, random_state=SEED, n_jobs=-1),
    "Random Forest — Dataset A (con outliers)"
)
results["RF_A"]  = metrics_rf_A
roc_data["RF_A"] = (y_test_A, proba_rf_A)

# ── Dataset B (sin outliers IQR) ────────────────────────────────────────────
model_dt_B, metrics_dt_B, _, proba_dt_B = train_full_pipeline(
    X_train_B, y_train_B, X_test_B, y_test_B,
    DecisionTreeClassifier(max_depth=10, min_samples_leaf=5, random_state=SEED),
    "Árbol de Decisión — Dataset B (sin outliers IQR)"
)
results["DT_B"]  = metrics_dt_B
roc_data["DT_B"] = (y_test_B, proba_dt_B)

model_rf_B, metrics_rf_B, _, proba_rf_B = train_full_pipeline(
    X_train_B, y_train_B, X_test_B, y_test_B,
    RandomForestClassifier(n_estimators=100, max_depth=10,
                           min_samples_leaf=5, random_state=SEED, n_jobs=-1),
    "Random Forest — Dataset B (sin outliers IQR)"
)
results["RF_B"]  = metrics_rf_B
roc_data["RF_B"] = (y_test_B, proba_rf_B)

## 13. Análisis ROC **[NUEVO]**

La **curva ROC** (*Receiver Operating Characteristic*) representa la tasa de verdaderos positivos (sensibilidad) frente a la tasa de falsos positivos (1 − especificidad) al variar el umbral de decisión. El **AUC-ROC** resume la curva en un único valor entre 0.5 (azar) y 1.0 (perfecto).

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
colors_roc = {"DT_A": "#4C72B0", "RF_A": "#DD8452",
              "DT_B": "#55A868", "RF_B": "#C44E52"}
labels_roc = {
    "DT_A": "Árbol Decisión — A (con outliers)",
    "RF_A": "Random Forest — A (con outliers)",
    "DT_B": "Árbol Decisión — B (sin outliers)",
    "RF_B": "Random Forest — B (sin outliers)",
}

for key, (y_test_k, proba_k) in roc_data.items():
    fpr, tpr, _ = roc_curve(y_test_k, proba_k)
    auc_val     = roc_auc_score(y_test_k, proba_k)
    ax.plot(fpr, tpr, color=colors_roc[key], linewidth=2,
            label=f"{labels_roc[key]} (AUC = {auc_val:.3f})")

ax.plot([0, 1], [0, 1], "--", color="grey", linewidth=1, label="Clasificador aleatorio")
ax.set_xlabel("Tasa de Falsos Positivos (1 − Especificidad)")
ax.set_ylabel("Tasa de Verdaderos Positivos (Sensibilidad)")
ax.set_title("Curvas ROC — comparación de modelos", fontsize=13)
ax.legend(loc="lower right", fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Matrices de confusión normalizadas para los 4 modelos
fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))
modelos = [
    ("DT_A", model_dt_A, X_test_A, y_test_A),
    ("RF_A", model_rf_A, X_test_A, y_test_A),
    ("DT_B", model_dt_B, X_test_B, y_test_B),
    ("RF_B", model_rf_B, X_test_B, y_test_B),
]
for ax, (key, mdl, X_t, y_t) in zip(axes, modelos):
    ConfusionMatrixDisplay.from_estimator(
        mdl, X_t, y_t, normalize="true", cmap="Blues",
        display_labels=["Sano", "Diabético"], ax=ax, colorbar=False
    )
    ax.set_title(labels_roc[key], fontsize=10)
plt.tight_layout()
plt.show()

## 14. CrossValidation y búsqueda de hiperparámetros

Aplicamos `GridSearchCV` (5-fold estratificado) sobre Random Forest + Dataset B, replicando la celda equivalente del notebook original.


In [ ]:
pipe_cv = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("clf",    RandomForestClassifier(random_state=SEED, n_jobs=-1)),
])

param_grid = {
    "clf__n_estimators": [50, 100],
    "clf__max_depth":    [5, 10],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

grid = GridSearchCV(
    estimator   = pipe_cv,
    param_grid  = param_grid,
    scoring     = "f1_weighted",
    cv          = cv,
    n_jobs      = -1,
    verbose     = 1,
)

print("Ejecutando 5-fold GridSearchCV (puede tardar)...")
grid.fit(X_train_B, y_train_B)

print(f"\nMejores parámetros: {grid.best_params_}")
print(f"Mejor F1 (CV)      : {grid.best_score_:.4f}")

metrics_cv, conf_cv, _, proba_cv = evaluar_modelo(
    grid.best_estimator_, X_test_B, y_test_B, "RF + CV (mejor modelo)"
)
print("\nMétricas CV (test set):")
for k, v in metrics_cv.items():
    if k != "Modelo":
        print(f"  {k:<12}: {v}")

results["RF_B_CV"]  = metrics_cv
roc_data["RF_B_CV"] = (y_test_B, proba_cv)

## 15. Importancia de variables — Random Forest

In [ ]:
# Extraer importancia del RF entrenado en Dataset B
rf_step      = model_rf_B.named_steps["clf"]
importances  = rf_step.feature_importances_

feat_importance = pd.DataFrame({
    "variable":    feature_cols,
    "importancia": importances,
}).sort_values("importancia", ascending=False)

print("Importancia de variables (Random Forest):")
print(feat_importance.to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(feat_importance["variable"][::-1],
               feat_importance["importancia"][::-1],
               color="steelblue", edgecolor="black")
ax.set_xlabel("Importancia (Gini)")
ax.set_title("Importancia de variables — Random Forest (Dataset B)", fontsize=13)
for bar, val in zip(bars, feat_importance["importancia"][::-1]):
    ax.text(bar.get_width() + 0.001,
            bar.get_y() + bar.get_height()/2,
            f"{val:.4f}", va="center", fontsize=9)
plt.tight_layout()
plt.show()

## 16. Tratamiento de datasets desbalanceados **[NUEVO]**

El dataset Pima tiene un desbalance moderado (~65 % sanos / 35 % diabéticos). Para problemas con desbalance fuerte (fraude, fallos, etc.) el modelo tiende a sesgarse hacia la clase mayoritaria. Veamos tres técnicas clásicas con `imbalanced-learn`:

| Técnica | Idea | Riesgo |
|--|--|--|
| **RandomUnderSampler** | Elimina ejemplos de la mayoría | Pérdida de información |
| **RandomOverSampler** | Duplica ejemplos de la minoría | Sobreajuste a duplicados |
| **SMOTE** | Sintetiza ejemplos minoritarios por interpolación entre vecinos k-NN | Puede generar ruido si las clases se solapan |

**Regla de oro:** el resampling se aplica **solo al conjunto de entrenamiento**, nunca al de test. Si lo combinas con CV, debe ir **dentro** del pipeline (con `imblearn.pipeline.Pipeline`) para evitar fugas entre folds.


In [ ]:
# Distribución de clases en el train ORIGINAL (sin resampling)
print("Distribución original del train (Dataset B):")
print(y_train_B.value_counts())
print(f"Ratio diabético/sano: {y_train_B.mean():.3f}")

### 16.1. Submuestreo aleatorio — `RandomUnderSampler`

In [ ]:
rus = RandomUnderSampler(random_state=SEED)
X_train_rus, y_train_rus = rus.fit_resample(X_train_B, y_train_B)

print("Distribución tras submuestreo:")
print(pd.Series(y_train_rus).value_counts())

### 16.2. Sobremuestreo aleatorio — `RandomOverSampler`

In [ ]:
ros = RandomOverSampler(random_state=SEED)
X_train_ros, y_train_ros = ros.fit_resample(X_train_B, y_train_B)

print("Distribución tras sobremuestreo:")
print(pd.Series(y_train_ros).value_counts())

### 16.3. SMOTE — *Synthetic Minority Over-sampling Technique*

In [ ]:
smote = SMOTE(random_state=SEED, k_neighbors=5)
X_train_smote, y_train_smote = smote.fit_resample(X_train_B, y_train_B)

print("Distribución tras SMOTE:")
print(pd.Series(y_train_smote).value_counts())

### 16.4. Comparación visual de las tres distribuciones

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
titulos = ["Original", "Undersampling", "Oversampling", "SMOTE"]
distribs = [y_train_B, y_train_rus, y_train_ros, y_train_smote]

for ax, t, y in zip(axes, titulos, distribs):
    counts = pd.Series(y).value_counts().sort_index()
    ax.bar(["Sano (0)", "Diabético (1)"], counts.values,
           color=["steelblue", "tomato"], edgecolor="black")
    ax.set_title(t, fontsize=11)
    ax.set_ylabel("Cantidad")
    for i, v in enumerate(counts.values):
        ax.text(i, v + 1, str(v), ha="center", fontsize=10, fontweight="bold")

plt.suptitle("Distribución de clases — antes y después del resampling",
             fontsize=13, y=1.05)
plt.tight_layout()
plt.show()

### 16.5. Entrenamiento y comparación con Random Forest

Comparamos el mismo Random Forest sobre los tres datasets re-muestreados frente al baseline (sin resampling).

**Importante:** usamos `ImbPipeline` para encadenar `scaler → sampler → clasificador`. Esto garantiza que el resampling se ejecuta **después** del scaler y **solo** durante el entrenamiento.


In [ ]:
def entrenar_con_sampler(sampler, nombre):
    """Entrena RF dentro de un pipeline con sampler y devuelve métricas + probas."""
    pipe = ImbPipeline(steps=[
        ("scaler",  StandardScaler()),
        ("sampler", sampler),
        ("clf",     RandomForestClassifier(n_estimators=100, max_depth=10,
                                           random_state=SEED, n_jobs=-1)),
    ])
    pipe.fit(X_train_B, y_train_B)
    metrics, conf, y_pred, y_proba = evaluar_modelo(pipe, X_test_B, y_test_B, nombre)
    return pipe, metrics, y_proba, conf

# Baseline: sin resampling
base_pipe   = build_preprocessing_pipeline(
    RandomForestClassifier(n_estimators=100, max_depth=10, random_state=SEED, n_jobs=-1)
)
base_pipe.fit(X_train_B, y_train_B)
metrics_base, conf_base, _, proba_base = evaluar_modelo(
    base_pipe, X_test_B, y_test_B, "RF — Baseline (sin resampling)"
)

# Con los tres samplers
_, metrics_rus, proba_rus, conf_rus = entrenar_con_sampler(
    RandomUnderSampler(random_state=SEED), "RF — Undersampling"
)
_, metrics_ros, proba_ros, conf_ros = entrenar_con_sampler(
    RandomOverSampler(random_state=SEED),  "RF — Oversampling"
)
_, metrics_sm,  proba_sm,  conf_sm  = entrenar_con_sampler(
    SMOTE(random_state=SEED),              "RF — SMOTE"
)

imb_results = pd.DataFrame([metrics_base, metrics_rus, metrics_ros, metrics_sm])
print("=" * 75)
print("COMPARACIÓN — técnicas de resampling")
print("=" * 75)
print(imb_results.to_string(index=False))

In [ ]:
# Curvas ROC comparativas: baseline vs los tres samplers
fig, ax = plt.subplots(figsize=(8, 7))
roc_imb = {
    "Baseline":      proba_base,
    "Undersampling": proba_rus,
    "Oversampling":  proba_ros,
    "SMOTE":         proba_sm,
}
colors_imb = {"Baseline": "#4C72B0", "Undersampling": "#DD8452",
              "Oversampling": "#55A868", "SMOTE": "#C44E52"}

for nombre, proba in roc_imb.items():
    fpr, tpr, _ = roc_curve(y_test_B, proba)
    auc_val     = roc_auc_score(y_test_B, proba)
    ax.plot(fpr, tpr, color=colors_imb[nombre], linewidth=2,
            label=f"{nombre} (AUC = {auc_val:.3f})")

ax.plot([0, 1], [0, 1], "--", color="grey", linewidth=1, label="Aleatorio")
ax.set_xlabel("Tasa de Falsos Positivos")
ax.set_ylabel("Tasa de Verdaderos Positivos")
ax.set_title("ROC — Efecto del resampling sobre Random Forest", fontsize=13)
ax.legend(loc="lower right", fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Matrices de confusión normalizadas
fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))
matrices = [
    ("Baseline",      conf_base),
    ("Undersampling", conf_rus),
    ("Oversampling",  conf_ros),
    ("SMOTE",         conf_sm),
]
for ax, (nombre, conf) in zip(axes, matrices):
    disp = ConfusionMatrixDisplay(conf, display_labels=["Sano", "Diabético"])
    disp.plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(nombre, fontsize=11)
plt.suptitle("Matrices de confusión — efecto del resampling", fontsize=13, y=1.05)
plt.tight_layout()
plt.show()

**Lectura del resultado**

- *Accuracy* suele caer ligeramente con resampling, pero **recall** sobre la clase minoritaria (diabéticos) mejora — que es lo que importa en un problema médico.
- **SMOTE** suele dar el mejor compromiso AUC/recall cuando las clases no se solapan mucho.
- En este dataset el desbalance es moderado, así que las diferencias son modestas; en problemas con desbalance fuerte (1:100 o más) el efecto es mucho más dramático.


## 17. Comparación final de modelos

In [ ]:
compare_pd = pd.DataFrame([
    results["DT_A"], results["RF_A"],
    results["DT_B"], results["RF_B"],
    results["RF_B_CV"],
]).set_index("Modelo")

print("=" * 80)
print("TABLA COMPARATIVA DE MODELOS")
print("=" * 80)
print(compare_pd.to_string())

In [ ]:
# Gráficos de barras por métrica
metrics_to_plot = ["Accuracy", "F1", "AUC-ROC"]
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

colors  = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B2"]
short_names = ["DT\n(A)", "RF\n(A)", "DT\n(B)", "RF\n(B)", "RF\nCV(B)"]

for ax, metric in zip(axes, metrics_to_plot):
    values = compare_pd[metric].values
    bars = ax.bar(short_names, values, color=colors, edgecolor="black", alpha=0.85)
    ax.set_ylim(0.5, 1.0)
    ax.set_title(metric, fontsize=13)
    ax.set_ylabel("Valor")
    ax.axhline(0.8, color="red", linestyle="--", alpha=0.5, label="0.8 referencia")
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f"{val:.3f}", ha="center", fontsize=9, fontweight="bold")
    ax.legend(fontsize=8)

plt.suptitle("Comparación de modelos por métrica", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 18. Persistencia del mejor modelo

En el ecosistema sklearn, el equivalente nativo es `joblib`. (MLflow también funciona con sklearn vía `mlflow.sklearn.log_model` si se desea trazabilidad de experimentos.)


In [ ]:
import joblib

best_model = grid.best_estimator_   # mejor pipeline del GridSearch
joblib.dump(best_model, "rf_best_diabetes.joblib")
print("Modelo guardado en rf_best_diabetes.joblib")

# Cargar de nuevo (verificación)
loaded = joblib.load("rf_best_diabetes.joblib")
print("\nMétricas tras recargar:")
metrics_loaded, _, _, _ = evaluar_modelo(loaded, X_test_B, y_test_B, "RF (recargado)")
for k, v in metrics_loaded.items():
    if k != "Modelo":
        print(f"  {k:<12}: {v}")